In [ ]:
import matplotlib.pyplot as plt
import torch
from math import ceil
from typing import List, Literal
from calculate_metrics import (
    celeb_runs,
    coco_runs
)
from pixel_arena.metrics import RunInfo

In [ ]:
model_code_names = {
    "gemini-pro": "gmn3",
    "gemini-pro-shuffled": "gmn3-shuffled",
    "gemini": "gmn2.5",
    "gpt": "gpti",
    "uni-moe-2-image": "um2i",
    "uni-moe-2-omni": "um2o",
}

In [ ]:
def draw_bar_chart(runs: List[RunInfo], metric_key: Literal["f1_scores", "iou_scores", "dice_scores"], metric_name: str):
    x_positions = list(range(len(runs)))
    bar_width = 0.25  # Make bar width smaller for 3 bars per group

    p1_x, p1_means, p1_stds = [], [], []
    p3_x, p3_means, p3_stds = [], [], []
    p5_x, p5_means, p5_stds = [], [], []

    for idx, run in enumerate(runs):
        metrics = getattr(run, metric_key)
        has_p3 = metrics.shape[1] >= 3
        has_p5 = metrics.shape[1] >= 5

        # Determine offsets for bar groups (centered around position)
        # If there's p3 and p5, positions will be -bar, 0, +bar; else, single at x
        if has_p3 and has_p5:
            p1_x.append(x_positions[idx] - bar_width)
            p3_x.append(x_positions[idx])
            p5_x.append(x_positions[idx] + bar_width)
        elif has_p3:  # This implicitly means p5 may be absent (should not happen given your stats, but for completeness)
            p1_x.append(x_positions[idx] - bar_width / 2)
            p3_x.append(x_positions[idx] + bar_width / 2)
        else:
            p1_x.append(x_positions[idx])

        p1_std, p1_mean = torch.std_mean(metrics[:, 0])
        p1_means.append(p1_mean.item())
        p1_stds.append(p1_std.item())

        if has_p3:
            p3_std, p3_mean = torch.std_mean(metrics[:, :3].max(dim=1).values)
            p3_means.append(p3_mean.item())
            p3_stds.append(p3_std.item())
        if has_p5:
            p5_std, p5_mean = torch.std_mean(metrics[:, :5].max(dim=1).values)
            p5_means.append(p5_mean.item())
            p5_stds.append(p5_std.item())

    fig_width = max(len(runs) * 1.2, 6)
    fig, ax = plt.subplots(figsize=(fig_width, 6))
    ax.bar(
        p1_x, p1_means, bar_width, yerr=p1_stds, capsize=5, label="P=1", color="#1f77b4"
    )
    if p3_means:
        ax.bar(
            p3_x,
            p3_means,
            bar_width,
            yerr=p3_stds,
            capsize=5,
            label="P=3",
            color="#ff7f0e",
        )
    if p5_means:
        ax.bar(
            p5_x,
            p5_means,
            bar_width,
            yerr=p5_stds,
            capsize=5,
            label="P=5",
            color="#2ca02c",
        )

    values_with_error = [mean + std for mean, std in zip(p1_means, p1_stds)]
    if p3_means:
        values_with_error.extend(mean + std for mean, std in zip(p3_means, p3_stds))
    if p5_means:
        values_with_error.extend(mean + std for mean, std in zip(p5_means, p5_stds))
    if values_with_error:
        max_ylim = ceil(max(values_with_error) / 0.1) * 0.1
        ax.set_ylim(0, max_ylim)

    ax.set_xticks(x_positions)
    code_names = []
    for run in runs:
        if run.model_name in model_code_names:
            code_names.append(model_code_names[run.model_name])
        else:
            code_names.append(run.model_name)
    ax.set_xticklabels(code_names, ha="center")
    ax.set_ylabel(f"{metric_name}")
    ax.set_title(f"{metric_name} with Std")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()

# Plot CelebAMask-HQ Results

In [ ]:
for run in celeb_runs:
    run.load_metrics()

draw_bar_chart(celeb_runs, "f1_scores", "F1 Score")

# Plot COCO Results

In [ ]:
for run in coco_runs:
    run.load_metrics()

draw_bar_chart(coco_runs, "f1_scores", "F1 Score")